In [13]:
import json
import os
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Aiproject'
file_name = 'combined_dataset (1).json'
file_path = os.path.join(folder_path, file_name)


contexts = []
if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                data = json.loads(line)
                if 'Context' in data:
                    contexts.append(data['Context'])
            except:
                continue





all_embeddings = model.encode(contexts, convert_to_tensor=True, device=device)

train_tensors, temp_tensors = train_test_split(all_embeddings, test_size=0.30, random_state=42)
test_tensors, val_tensors = train_test_split(temp_tensors, test_size=0.50, random_state=42)

class PsychologyNet(nn.Module):
    def __init__(self, input_dim):
        super(PsychologyNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        return self.fc(x)

input_dim = train_tensors.shape[1]
net = PsychologyNet(input_dim).to(device)
optimizer = optim.Adam(net.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(train_tensors), batch_size=32, shuffle=True)

def run_training_cycle(epochs=20):
    for epoch in range(epochs):
        net.train()
        epoch_loss = 0
        for batch in train_loader:
            inputs = batch[0].to(device)
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        net.eval()
        with torch.no_grad():
            test_outputs = net(test_tensors)
            test_loss = criterion(test_outputs, test_tensors)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_loss/len(train_loader):.4f} - Test Loss: {test_loss.item():.4f}")

run_training_cycle(epochs=20)

print("\nRunning Final Validation...")
net.eval()
with torch.no_grad():
    val_outputs = net(val_tensors)
    val_loss = criterion(val_outputs, val_tensors)
    print(f"Final Validation Loss: {val_loss.item():.4f}")

torch.save(net.state_dict(), "psychology_model.pth")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/20 - Train Loss: 0.0017 - Test Loss: 0.0011
Epoch 2/20 - Train Loss: 0.0008 - Test Loss: 0.0007
Epoch 3/20 - Train Loss: 0.0005 - Test Loss: 0.0005
Epoch 4/20 - Train Loss: 0.0004 - Test Loss: 0.0004
Epoch 5/20 - Train Loss: 0.0004 - Test Loss: 0.0004
Epoch 6/20 - Train Loss: 0.0003 - Test Loss: 0.0003
Epoch 7/20 - Train Loss: 0.0003 - Test Loss: 0.0003
Epoch 8/20 - Train Loss: 0.0003 - Test Loss: 0.0003
Epoch 9/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 10/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 11/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 12/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 13/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 14/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 15/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 16/20 - Train Loss: 0.0002 - Test Loss: 0.0002
Epoch 17/20 - Train Loss: 0.0002

In [ ]:
from google.colab import drive
drive.mount('/content/drive')